# 6. Sorting

- `np.sort(a)` returns a sorted **copy**
- `a.sort()` sorts **in place** and returns `None`
- `np.argsort(a)` returns the indices that would sort the array

In [1]:
import numpy as np

rng = np.random.default_rng(1)
arr = rng.integers(0, 50, size=(4, 5))
arr

array([[23, 25, 37, 47,  1],
       [ 7, 41, 47, 12, 15],
       [43, 21, 13, 41, 12],
       [20, 32, 27,  4,  1]])

## Sorting along an axis

The default is `axis=-1`, the last axis, so for a 2-D array each **row** gets sorted on its own. `axis=0` sorts each **column**. And `axis=None` flattens first.

Note that each row or column is sorted independently, so the values in a row don't stay together when you sort by column. If you want to sort whole rows by one column, see the `argsort` section below.

In [2]:
print(np.sort(arr))            # each row sorted
print(np.sort(arr, axis=0))    # each column sorted
print(np.sort(arr, axis=None)) # everything, flattened

[[ 1 23 25 37 47]
 [ 7 12 15 41 47]
 [12 13 21 41 43]
 [ 1  4 20 27 32]]
[[ 7 21 13  4  1]
 [20 25 27 12  1]
 [23 32 37 41 12]
 [43 41 47 47 15]]
[ 1  1  4  7 12 12 13 15 20 21 23 25 27 32 37 41 41 43 47 47]


In [3]:
b = arr.copy()
result = b.sort(axis=1)
print(result)    # None! sort() works in place
print(b)

None
[[ 1 23 25 37 47]
 [ 7 12 15 41 47]
 [12 13 21 41 43]
 [ 1  4 20 27 32]]


## Descending order

There's no `reverse=True` option. Reverse the result, or for numbers, sort the negatives:

In [4]:
x = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print(np.sort(x)[::-1])
print(-np.sort(-x))

[9 6 5 4 3 2 1 1]
[9 6 5 4 3 2 1 1]


## `argsort`: sort one thing by another

`argsort` is more useful than it looks. Once you have the order, you can apply it to other arrays too.

In [5]:
names = np.array(['Dana', 'Ali', 'Chen', 'Bo'])
scores = np.array([88, 95, 72, 91])

order = np.argsort(scores)[::-1]     # highest score first
for name, score in zip(names[order], scores[order]):
    print(f'{name:5} {score}')

Ali   95
Bo    91
Dana  88
Chen  72


Sorting the rows of a table by one column:

In [6]:
print(arr)
print(arr[arr[:, 2].argsort()])   # rows ordered by the third column

[[23 25 37 47  1]
 [ 7 41 47 12 15]
 [43 21 13 41 12]
 [20 32 27  4  1]]
[[43 21 13 41 12]
 [20 32 27  4  1]
 [23 25 37 47  1]
 [ 7 41 47 12 15]]


## Sorting by several keys

`np.lexsort` sorts by multiple keys. The **last** key you pass is the primary one, which is easy to get backwards.

In [7]:
last  = np.array(['Smith', 'Jones', 'Smith', 'Adams'])
first = np.array(['Zoe',   'Amy',   'Bob',   'Cal'])

idx = np.lexsort((first, last))   # by last name, then first name
np.column_stack((last[idx], first[idx]))

array([['Adams', 'Cal'],
       ['Jones', 'Amy'],
       ['Smith', 'Bob'],
       ['Smith', 'Zoe']], dtype='<U5')

## Top-k without a full sort

If you only need the k smallest (or largest) values, `np.partition` is faster than sorting everything. It guarantees the k-th element is in its sorted position, with smaller values before it and larger ones after, but those two groups aren't sorted themselves.

In [8]:
x = rng.integers(0, 1000, size=20)
k = 3
print(np.partition(x, k)[:k])           # 3 smallest, in no particular order
print(np.sort(np.partition(x, -k)[-k:]))  # 3 largest, sorted afterwards

[123 124 134]
[865 903 976]


In [9]:
idx = np.argpartition(x, -k)[-k:]       # positions of the 3 largest
idx, x[idx]

(array([ 0, 16, 12]), array([865, 903, 976]))

## Searching in a sorted array

`np.searchsorted` finds where values would need to be inserted to keep the array sorted. It's a binary search, so it's fast even on huge arrays, and it's handy for binning:

In [10]:
edges = np.array([0, 18, 65, 120])
ages = np.array([5, 18, 30, 70])
labels = np.array(['child', 'adult', 'senior'])
labels[np.searchsorted(edges, ages, side='right') - 1]

array(['child', 'adult', 'adult', 'senior'], dtype='<U6')